# M2 v2 — Sequential Fine-Tuning (All 6 Orderings)

## Core idea
Instead of combining losses simultaneously (gradient interference), train **sequentially**: 
train with loss 1 → save → fine-tune with loss 2 → save → fine-tune with loss 3.

## Losses (chosen by Phase 1 performance)
- **L_dec (E7)**: Decision-only supervision. Best composite-5, CC, reasoning_mean.
- **L_wsft (E1)**: Section-weighted SFT. Most balanced, best format compliance.
- **L_ent (E5b)**: Explanation entropy matching. Best safety among entropy methods, strong CC.

## All 6 orderings
| Name | Stage 1 | Stage 2 | Stage 3 | Hypothesis |
|------|---------|---------|---------|------------|
| DWE | L_dec | L_wsft | L_ent | Reasoning foundation → format → calibration |
| DEW | L_dec | L_ent | L_wsft | Reasoning → calibration → format polish |
| WDE | L_wsft | L_dec | L_ent | Format first → reasoning → calibration |
| WED | L_wsft | L_ent | L_dec | Format → calibration → reasoning |
| EDW | L_ent | L_dec | L_wsft | Calibration → reasoning → format |
| EWD | L_ent | L_wsft | L_dec | Calibration → format → reasoning |

## Training budget
- **1 epoch per stage × 3 stages = 3 epochs total** (50% more than Phase 1's 2 epochs)
- Decreasing LR: Stage 1 = 2e-4, Stage 2 = 7e-5, Stage 3 = 2e-5
- Each stage loads the previous stage's LoRA checkpoint and continues training

In [1]:
# Cell 0: Environment + Model Selection (Linux/WSL2)
import os, torch

TRAIN_MODEL = "qwen25_1p5b_base"
#TRAIN_MODEL = "qwen25_3b_base"
#TRAIN_MODEL = "qwen25_7b_base"

use_bf16 = "7b" not in TRAIN_MODEL

print(f"GPU: {torch.cuda.get_device_name()}")
print(f"VRAM: {torch.cuda.get_device_properties(0).total_mem / 1024**3:.1f} GB")
print(f"Will train: {TRAIN_MODEL} | bf16={use_bf16}")

Will train: qwen25_1p5b_base | bf16=True
⚠️ Restart kernel before switching to a different model


In [2]:
# Cell 1: Imports + data
import os, sys, json, math, re, itertools
from typing import List, Dict, Any, Tuple
from tqdm import tqdm

import torch
from torch.utils.data import Dataset
from transformers import TrainingArguments, Trainer
from peft import PeftModel
from transformers import AutoTokenizer, AutoModelForCausalLM

sys.path.insert(0, os.path.expanduser(""))
from shared_utils import (
    RUNS_DIR, MAX_SEQ_LEN, LR, SEED, LAMBDA,
    W_FORMAT, W_DECISION, W_EXPL,
    STUDENTS, load_data, load_student,
    get_section_spans, in_any_span,
    compute_mean_confidence,
    find_decision_span_chars, find_expl_span_chars,
    teacher_section_entropy_mean,
    build_prompt_text, tokenize_and_mask, FlexibleCollator,
)

OUT_DIR = os.path.join(RUNS_DIR, "m2v2_sequential")
os.makedirs(OUT_DIR, exist_ok=True)

# ── Sequential orderings ──
ORDERINGS = {
    "DWE": ["dec", "wsft", "ent"],
    "DEW": ["dec", "ent", "wsft"],
    "WDE": ["wsft", "dec", "ent"],
    "WED": ["wsft", "ent", "dec"],
    "EDW": ["ent", "dec", "wsft"],
    "EWD": ["ent", "wsft", "dec"],
}

# Decreasing LR across stages
STAGE_LR = {0: 2e-4, 1: 7e-5, 2: 2e-5}
EPOCHS_PER_STAGE = 1  # 1 epoch per stage × 3 stages = 3 total

prompts, teacher_rows = load_data()
MEAN_C = compute_mean_confidence(teacher_rows)
print(f"Mean confidence: {MEAN_C:.6f}")
print(f"Orderings to run: {list(ORDERINGS.keys())}")

c:\Users\adishalit1\AppData\Local\anaconda3\envs\kd\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


✅ Loaded 5000 teacher samples from: data\clinical_pharm_teacher_train_6000.jsonl
Mean confidence: 0.999763
Orderings to run: ['DWE', 'DEW', 'WDE', 'WED', 'EDW', 'EWD']


In [3]:
# Cell 2: Dataset — precomputes all fields
class M2v2DatasetFast(Dataset):
    def __init__(self, rows, prompts, tokenizer, is_instruct, mean_c):
        print("  Precomputing dataset...")
        self.items = []
        for idx in tqdm(range(len(rows)), desc="  Tokenizing"):
            r = rows[idx]
            prompt = prompts[r["id"]]
            answer = r["teacher_text"]

            prompt_text = build_prompt_text(tokenizer, prompt, is_instruct)
            input_ids, offsets, labels, answer_start = tokenize_and_mask(
                tokenizer, prompt_text, answer
            )

            # WSFT weights
            wsft_weights = [0.0] * len(input_ids)
            spans = get_section_spans(answer)
            dec_spans = [(answer_start + s, answer_start + e) for s, e in spans["decision"]]
            expl_spans = [(answer_start + s, answer_start + e) for s, e in spans["explanation"]]
            for i, (s, e) in enumerate(offsets):
                if e <= s: continue
                if s >= answer_start:
                    w = W_FORMAT
                    if in_any_span(s, e, dec_spans): w = W_DECISION
                    if in_any_span(s, e, expl_spans): w = W_EXPL
                    wsft_weights[i] = float(w)
            active_w = [w for w in wsft_weights if w > 0]
            if active_w:
                mw = sum(active_w) / len(active_w)
                if mw > 1e-6:
                    wsft_weights = [w / mw if w > 0 else 0.0 for w in wsft_weights]

            # Decision mask
            dec_mask = torch.zeros(len(input_ids), dtype=torch.bool)
            dec_span_chars = find_decision_span_chars(answer)
            if dec_span_chars:
                ds, de = dec_span_chars
                for i, (s, e) in enumerate(offsets):
                    if labels[i] != -100 and not (e <= answer_start + ds or s >= answer_start + de):
                        dec_mask[i] = True

            # Explanation mask
            expl_mask = torch.zeros(len(input_ids), dtype=torch.bool)
            expl_span_chars = find_expl_span_chars(answer)
            if expl_span_chars:
                es, ee = expl_span_chars
                for i, (s, e) in enumerate(offsets):
                    if labels[i] != -100 and not (e <= answer_start + es or s >= answer_start + ee):
                        expl_mask[i] = True

            # Teacher entropy on explanation
            ent_teacher_expl = teacher_section_entropy_mean(r, expl_span_chars)

            self.items.append({
                "input_ids": torch.tensor(input_ids, dtype=torch.long),
                "attention_mask": torch.ones(len(input_ids), dtype=torch.long),
                "labels": torch.tensor(labels, dtype=torch.long),
                "wsft_weights": torch.tensor(wsft_weights, dtype=torch.float),
                "dec_mask": dec_mask,
                "expl_mask": expl_mask,
                "ent_teacher_expl": ent_teacher_expl.float(),
            })
        print(f"  ✅ Precomputed {len(self.items)} samples")

    def __len__(self): return len(self.items)
    def __getitem__(self, idx): return self.items[idx]

print("✅ Dataset class ready")

✅ Dataset class ready


In [4]:
# Cell 3: Three single-objective trainers
class DecOnlyTrainer(Trainer):
    """E7-style: loss on decision tokens only."""
    def __init__(self, *args, **kwargs):
        super().__init__(*args, **kwargs)
        self._ce = torch.nn.CrossEntropyLoss(reduction="none")

    def compute_loss(self, model, inputs, return_outputs=False, **kwargs):
        labels = inputs.pop("labels")
        _ = inputs.pop("wsft_weights"); _ = inputs.pop("expl_mask"); _ = inputs.pop("ent_teacher_expl")
        dec_mask = inputs.pop("dec_mask")

        logits = model(**inputs).logits
        sl = logits[:, :-1, :].contiguous()
        ll = labels[:, 1:].contiguous()
        dm = dec_mask[:, 1:]
        V = sl.size(-1)

        tok_loss = self._ce(sl.view(-1, V), ll.view(-1)).view(ll.size())
        active = (ll != -100).float()
        da = dm.float() * active
        loss = (tok_loss * da).sum() / da.sum().clamp(min=1.0)
        return (loss, model(input_ids=inputs["input_ids"], attention_mask=inputs["attention_mask"])) if return_outputs else loss


class WsftTrainer(Trainer):
    """E1-style: section-weighted SFT."""
    def __init__(self, *args, **kwargs):
        super().__init__(*args, **kwargs)
        self._ce = torch.nn.CrossEntropyLoss(reduction="none")

    def compute_loss(self, model, inputs, return_outputs=False, **kwargs):
        labels = inputs.pop("labels")
        wsft_w = inputs.pop("wsft_weights")
        _ = inputs.pop("dec_mask"); _ = inputs.pop("expl_mask"); _ = inputs.pop("ent_teacher_expl")

        logits = model(**inputs).logits
        sl = logits[:, :-1, :].contiguous()
        ll = labels[:, 1:].contiguous()
        sw = wsft_w[:, 1:].contiguous()
        V = sl.size(-1)

        tok_loss = self._ce(sl.view(-1, V), ll.view(-1)).view(ll.size())
        active = (ll != -100).float()
        w = sw * active
        denom = active.sum(dim=1).clamp(min=1.0)
        loss = ((tok_loss * w).sum(dim=1) / denom).mean()
        return (loss, model(input_ids=inputs["input_ids"], attention_mask=inputs["attention_mask"])) if return_outputs else loss


class ExplEntropyTrainer(Trainer):
    """E5b-style: explanation entropy matching + base SFT."""
    def __init__(self, *args, **kwargs):
        super().__init__(*args, **kwargs)
        self._ce = torch.nn.CrossEntropyLoss(reduction="none")

    def compute_loss(self, model, inputs, return_outputs=False, **kwargs):
        labels = inputs.pop("labels")
        wsft_w = inputs.pop("wsft_weights")
        _ = inputs.pop("dec_mask")
        expl_mask = inputs.pop("expl_mask")
        ent_teacher = inputs.pop("ent_teacher_expl")

        logits = model(**inputs).logits
        sl = logits[:, :-1, :].contiguous()
        ll = labels[:, 1:].contiguous()
        sw = wsft_w[:, 1:].contiguous()
        em = expl_mask[:, 1:]
        B, S, V = sl.size()

        tok_loss = self._ce(sl.view(-1, V), ll.view(-1)).view(B, S)
        active = (ll != -100).float()

        # Base SFT component (keeps format)
        w = sw * active
        denom = active.sum(dim=1).clamp(min=1.0)
        L_sft = ((tok_loss * w).sum(dim=1) / denom).mean()

        # Entropy matching on explanation tokens
        expl_active = em & (ll != -100)
        ent_student = torch.zeros(B, device=sl.device)
        for b in range(B):
            idx = expl_active[b].nonzero(as_tuple=True)[0]
            if len(idx) > 0:
                p = torch.softmax(sl[b, idx, :], dim=-1)
                ent_student[b] = -(p * torch.log(p + 1e-9)).sum(-1).mean()
        L_ent = LAMBDA * ((ent_student - ent_teacher.to(sl.device)) ** 2).mean()

        loss = L_sft + L_ent
        return (loss, model(input_ids=inputs["input_ids"], attention_mask=inputs["attention_mask"])) if return_outputs else loss


TRAINER_MAP = {
    "dec": DecOnlyTrainer,
    "wsft": WsftTrainer,
    "ent": ExplEntropyTrainer,
}
LOSS_DISPLAY = {"dec": "L_dec_only (E7)", "wsft": "L_wsft (E1)", "ent": "L_expl_ent (E5b)"}

print("✅ All three trainers ready")

✅ All three trainers ready


In [ ]:
# Cell 4: Sequential training — all 6 orderings
from peft import PeftModel
from transformers import AutoTokenizer, AutoModelForCausalLM
import torch, gc

cfg = STUDENTS[TRAIN_MODEL]

# Precompute dataset once
tokenizer_ref = AutoTokenizer.from_pretrained(cfg["path"], trust_remote_code=True)
if tokenizer_ref.pad_token is None: tokenizer_ref.pad_token = tokenizer_ref.eos_token
dataset = M2v2DatasetFast(teacher_rows, prompts, tokenizer_ref, cfg["is_instruct"], MEAN_C)
collator = FlexibleCollator(
    tokenizer_ref,
    extra_1d_fields=["wsft_weights", "dec_mask", "expl_mask"],
    extra_scalar_fields=["ent_teacher_expl"],
)

# Check which orderings already done
done = set()
for order_name in ORDERINGS:
    marker = os.path.join(OUT_DIR, TRAIN_MODEL, order_name, "stage3", "adapter_config.json")
    if os.path.exists(marker):
        done.add(order_name)
        print(f"⏩ {order_name} already done")

remaining = [o for o in ORDERINGS if o not in done]
print(f"\nWill run {len(remaining)} orderings: {remaining}")

for order_name in remaining:
    stages = ORDERINGS[order_name]
    print(f"\n{'='*70}")
    print(f"  M2v2: {TRAIN_MODEL} / {order_name} = {' → '.join(LOSS_DISPLAY[s] for s in stages)}")
    print(f"{'='*70}")

    order_dir = os.path.join(OUT_DIR, TRAIN_MODEL, order_name)

    prev_adapter_path = None

    for stage_idx, loss_key in enumerate(stages):
        stage_name = f"stage{stage_idx + 1}"
        stage_dir = os.path.join(order_dir, stage_name)
        os.makedirs(stage_dir, exist_ok=True)

        # Check if this stage is already done
        if os.path.exists(os.path.join(stage_dir, "adapter_config.json")):
            print(f"  ⏩ {stage_name} ({LOSS_DISPLAY[loss_key]}) already done")
            prev_adapter_path = stage_dir
            continue

        lr = STAGE_LR[stage_idx]
        print(f"\n  ── {stage_name}: {LOSS_DISPLAY[loss_key]} | LR={lr:.1e} | {EPOCHS_PER_STAGE} epoch ──")

        if prev_adapter_path is not None:
            # Load base model + previous stage's LoRA
            print(f"    Loading checkpoint from {prev_adapter_path}...")
            tokenizer = AutoTokenizer.from_pretrained(cfg["path"], trust_remote_code=True)
            if tokenizer.pad_token is None: tokenizer.pad_token = tokenizer.eos_token

            if "7b" in TRAIN_MODEL:
                from transformers import BitsAndBytesConfig
                bnb = BitsAndBytesConfig(load_in_8bit=True)
                base = AutoModelForCausalLM.from_pretrained(
                    cfg["path"], quantization_config=bnb, device_map="auto", trust_remote_code=True)
            else:
                base = AutoModelForCausalLM.from_pretrained(
                    cfg["path"], torch_dtype=torch.bfloat16, device_map="auto", trust_remote_code=True)

            model = PeftModel.from_pretrained(base, prev_adapter_path, is_trainable=True)
            model.train()
            trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
            print(f"    ✅ Loaded with {trainable:,} trainable params")
        else:
            # First stage: fresh LoRA
            tokenizer, model = load_student(TRAIN_MODEL, cfg)

        TrainerClass = TRAINER_MAP[loss_key]
        trainer = TrainerClass(
            model=model,
            args=TrainingArguments(
                output_dir=stage_dir,
                num_train_epochs=EPOCHS_PER_STAGE,
                per_device_train_batch_size=8,
                gradient_accumulation_steps=4,
                learning_rate=lr,
                logging_steps=25,
                save_strategy="no",
                bf16=use_bf16,
                seed=SEED + stage_idx,
                report_to="none",
                remove_unused_columns=False,
                dataloader_num_workers=4,
                label_smoothing_factor=0.1,
                gradient_checkpointing="7b" in TRAIN_MODEL,
            ),
            train_dataset=dataset,
            data_collator=collator,
        )

        trainer.train()
        model.save_pretrained(stage_dir)
        tokenizer.save_pretrained(stage_dir)
        print(f"    ✅ Saved {stage_name} → {stage_dir}")

        prev_adapter_path = stage_dir

        del model, tokenizer, trainer
        if "base" in dir(): del base
        gc.collect()
        torch.cuda.empty_cache()

    # Copy final stage as the "result" adapter
    final_dir = os.path.join(order_dir, "final")
    if not os.path.exists(final_dir):
        import shutil
        shutil.copytree(prev_adapter_path, final_dir)
        print(f"  ✅ Final model: {final_dir}")

print(f"\n✅ M2v2 complete for {TRAIN_MODEL}.")

  Precomputing dataset...


  Tokenizing: 100%|██████████| 5000/5000 [00:22<00:00, 225.17it/s]


  ✅ Precomputed 5000 samples

Will run 6 orderings: ['DWE', 'DEW', 'WDE', 'WED', 'EDW', 'EWD']

  M2v2: qwen25_1p5b_base / DWE = L_dec_only (E7) → L_wsft (E1) → L_expl_ent (E5b)

  ── stage1: L_dec_only (E7) | LR=2.0e-04 | 1 epoch ──
  Loading qwen25_1p5b_base from Qwen/Qwen2.5-1.5B...


`torch_dtype` is deprecated! Use `dtype` instead!
Loading weights: 100%|██████████| 338/338 [00:00<00:00, 401.91it/s]


  ✅ qwen25_1p5b_base: 4,358,144 trainable / 1,548,072,448 total params


...

Loading weights: 100%|██████████| 338/338 [00:00<00:00, 468.87it/s]


    ✅ Loaded with 4,358,144 trainable params


Step,Training Loss
25,49.900684
50,43.916646
75,41.877485
100,40.786824
125,39.949934


SystemError: unknown opcode

: 